In [5]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import os
import requests
from bs4 import BeautifulSoup

ticker = "AAPL"   
data = yf.download(ticker, period="1mo")
data = data.reset_index()  
data.head()

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2026-07-28,339.786926,342.594533,335.310806,339.736982,51859000
1,2026-07-29,337.898590,344.273097,337.059318,339.437272,56090800
2,2026-07-30,333.142670,334.461540,329.305982,332.812967,74817800
3,2026-07-31,308.643829,310.422294,299.741503,304.547356,132489100
4,2026-08-03,303.158569,311.531323,302.299295,309.313235,75052000


In [4]:
os.makedirs('data/raw', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d-%H%M')
filename = f'data/raw/api_yfinance_{ticker}_{timestamp}.csv'
data.to_csv(filename, index=False)
print("Saved to", filename)

Saved to data/raw/api_yfinance_AAPL_20260827-2320.csv


In [7]:

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}
response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

table = soup.find('table', {'id': 'constituents'})
table is not None

True

In [8]:
rows = table.find_all('tr')
headers = [th.text.strip() for th in rows[0].find_all('th')]

table_data = []
for row in rows[1:]:
    cells = [td.text.strip() for td in row.find_all('td')]
    if cells:
        table_data.append(cells)

scrape_df = pd.DataFrame(table_data, columns=headers)
scrape_df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [9]:
print("Shape:", scrape_df.shape)
print("Null counts:\n", scrape_df.isna().sum())
print("Sample dtypes:\n", scrape_df.dtypes)

Shape: (503, 8)
Null counts:
 Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64
Sample dtypes:
 Symbol                   object
Security                 object
GICS Sector              object
GICS Sub-Industry        object
Headquarters Location    object
Date added               object
CIK                      object
Founded                  object
dtype: object


In [10]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M')
filename = f'data/raw/scrape_wikipedia_sp500_{timestamp}.csv'
scrape_df.to_csv(filename, index=False)
print("Saved to", filename)

Saved to data/raw/scrape_wikipedia_sp500_20260827-2323.csv


## Sources
- API: Yahoo Finance via yfinance, ticker AAPL, 1-month daily prices
- Scrape: Wikipedia "List of S&P 500 companies" — constituents table

## Assumptions & Risks
- yfinance depends on an unofficial API that could change or break
- Wikipedia table structure could change, breaking the scraper
- No authentication was required for either source